# 04 — Embedding Study
## Hybrid Quantum-Classical NLP Framework

**⚠️ Run Cell 1 → Restart Runtime → Run Cell 2 onwards**

In [ ]:
# ============================================================
# CELL 1 — Installation  (then Runtime → Restart Runtime)
# ============================================================
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.strip())
    return r

print("Step 1/6: Wiping stale packages from disk...")
run("pip uninstall -y pennylane pennylane-lightning autoray jax jaxlib 2>/dev/null || true")
run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 -name 'pennylane*' -exec rm -rf {} + 2>/dev/null || true")
run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 -name 'PennyLane*' -exec rm -rf {} + 2>/dev/null || true")
run("find /usr/local/lib/python3.12/dist-packages -maxdepth 1 -name 'autoray*'   -exec rm -rf {} + 2>/dev/null || true")
print("  ✓ Stale files wiped")

print("Step 2/6: Installing numpy + scipy (ABI-compatible pair)...")
run("pip install -q --no-cache-dir numpy==1.26.4")
run("pip install -q --no-cache-dir scipy==1.13.1")
print("  ✓ numpy/scipy OK")

print("Step 3/6: Pinning JAX to a pennylane-compatible version...")
run("pip install -q --no-cache-dir 'jax==0.4.26' 'jaxlib==0.4.26'")
print("  ✓ JAX pinned")

print("Step 4/6: Installing autoray + pennylane (matched pair)...")
run("pip install -q --no-cache-dir autoray==0.6.12")
run("pip install -q --no-cache-dir pennylane==0.38.0 pennylane-lightning==0.38.0")
print("  ✓ pennylane/autoray OK")

print("Step 5/6: Installing NLP/ML stack...")
run("pip install -q --no-cache-dir "
    "numba>=0.57.1 "
    "transformers>=4.40.0 "
    "datasets>=2.19.0 "
    "scikit-learn>=1.4.0 "
    "gensim>=4.3.0 "
    "nltk>=3.8.0 "
    "seaborn>=0.13.0 "
    "tabulate>=0.9.0 "
    "pyyaml>=6.0 "
    "pandas==2.2.2 "
    "requests==2.32.4 "
    "'rich>=12.4.4,<14' "
    "'cachetools>=4.2.1,<7' "
    "'pillow<12.0' "
    "'tomlkit>=0.12.0,<0.14.0'"
)
print("  ✓ NLP/ML stack OK")

print("Step 6/6: Verifying installed versions...")
for pkg in ["pennylane", "autoray", "numpy", "scipy", "jax"]:
    run(f"pip show {pkg} | grep -E 'Name|Version'")

print()
print("=" * 60)
print("✅ Done! NOW: Runtime → Restart Runtime")
print("   Then run Cell 2 onwards.")
print("=" * 60)


In [ ]:
# ============================================================
# CELL 2 — Verify Imports  (AFTER restarting runtime)
# ============================================================
import importlib

print("Verifying all imports...")
print("-" * 50)

ok = True
checks = [
    ("numpy",       "1.26"),
    ("scipy",       "1.13"),
    ("jax",         "0.4.26"),
    ("autoray",     "0.6"),
    ("pennylane",   "0.38"),
    ("torch",       None),
    ("transformers",None),
    ("sklearn",     None),
]

for mod, prefix in checks:
    try:
        m   = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        chk = prefix is None or ver.startswith(prefix)
        print(f"  {'✅' if chk else '⚠️ '} {mod:<15} {ver}")
        if not chk: ok = False
    except Exception as e:
        print(f"  ❌ {mod:<15} {e}")
        ok = False

print()
try:
    import pennylane as qml
    dev = qml.device("default.qubit", wires=2)
    @qml.qnode(dev)
    def _t():
        qml.Hadamard(wires=0)
        qml.CNOT(wires=[0,1])
        return qml.expval(qml.PauliZ(0))
    print(f"  ✅ pennylane circuit  result={float(_t()):.4f}")
except Exception as e:
    print(f"  ❌ pennylane circuit  {e}")
    ok = False

print()
if ok:
    print("✅ All checks passed — safe to continue!")
else:
    print("❌ Checks failed.")
    print("   → Runtime → Factory Reset Runtime, then rerun Cell 1.")


In [ ]:
# ============================================================
# CELL 3 — Mount Google Drive
# ============================================================
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/HybridQuantumNLP"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f"✅ Drive mounted. Project dir: {DRIVE_PROJECT_DIR}")


In [ ]:
# ============================================================
# CELL 4A — Load Project: Upload ZIP  (run this OR 4B)
# ============================================================
from google.colab import files
import zipfile, glob, os, sys

print("Upload your project zip...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
PROJECT_DIR = "/content/project"

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(PROJECT_DIR)

run_files = glob.glob(f"{PROJECT_DIR}/**/run.py", recursive=True)
if run_files:
    PROJECT_DIR = os.path.dirname(run_files[0])

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"✅ Project loaded: {PROJECT_DIR}")


In [ ]:
# ============================================================
# CELL 4B — Load Project: From Drive  (run this OR 4A)
# ============================================================
import os, sys, shutil, zipfile

DRIVE_ZIP_PATH = "/content/drive/MyDrive/HybridQuantumNLP/JU_project.zip"
PROJECT_DIR    = "/content/project"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as z:
    z.extractall(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"✅ Project loaded from Drive: {PROJECT_DIR}")


In [ ]:
# ============================================================
# CELL 5 — Configuration
# ============================================================
import torch

LANGUAGE         = "english"    # "english", "hindi", "bengali"
EMBEDDING        = "tfidf"      # will be overridden per experiment
N_QUBITS         = 4
N_LAYERS         = 2
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
MAX_SAMPLES      = None          # None = full dataset, e.g. 500 for quick test
QUANTUM_EPOCHS   = 30

print(f"Language:    {LANGUAGE}")
print(f"Qubits:      {N_QUBITS}")
print(f"Device:      {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Max samples: {MAX_SAMPLES or 'Full dataset'}")


In [ ]:
# ============================================================
# CELL 6 — Build Config Object
# ============================================================
from config import load_config

cfg = load_config(overrides={
    "device": DEVICE,
    "quantum_template": {"n_qubits": N_QUBITS, "n_layers": N_LAYERS, "epochs": QUANTUM_EPOCHS},
    "custom_vqc":       {"n_qubits": N_QUBITS, "n_layers": N_LAYERS, "epochs": QUANTUM_EPOCHS},
    "embeddings": {
        "method": EMBEDDING,
        "indicbert": {"model_name": "ai4bharat/IndicBERTv2-MLM-only"}
    }
})
print("✅ Config ready.")


In [ ]:
# ============================================================
# CELL 7 — Dataset Overview
# ============================================================
from data.registry import get_dataset_info, UNIFIED_LABELS

print("=" * 65)
for name, lang in [("go_emotions","English"),("emo_in_hindi","Hindi"),("emonoba","Bengali")]:
    info = get_dataset_info(name)
    src  = info.hf_id or info.github_url or info.path
    print(f"  {lang}: {info.name}")
    print(f"    Source: {info.source} ({src})")
    print(f"    Labels: {info.num_labels} → {info.label_names[:6]}")
print(f"  Unified labels: {UNIFIED_LABELS}")
print("=" * 65)


In [ ]:
# ============================================================
# CELL 8 — Dataset Load Test
# ============================================================
from data.loader import load_dataset_for_language

test_ds = load_dataset_for_language(LANGUAGE, cfg, max_samples=100)
print(test_ds.summary())
print(f"Labels: {test_ds.label_names}")
print(f"Sample: {test_ds.train['texts'][0][:120]}...")
print("✅ Dataset OK")


---
## Embedding Study

**Fixed:**
- Custom VQC-v2 (data re-uploading)
- tanh activation
- Adam optimizer (framework default)

**Variable:**
- TF-IDF (sparse, 5000-d)
- mBERT (bert-base-multilingual-cased, 768-d)
- IndicBERT (IndicBERTv2-MLM-only, 768-d)

**Metrics collected per embedding:**
- Accuracy, Precision, Recall, F1
- Classification report (per-class)
- Confusion matrix
- Training time & parameter count
- Sample predictions for qualitative analysis

In [ ]:
# ============================================================
# CELL 9 — Model Validation (Public Access Test)
# ============================================================
from transformers import AutoTokenizer, AutoModel

model_name = "ai4bharat/IndicBERTv2-MLM-only"
try:
    print(f"Loading tokenizer for {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(f"Loading model for {model_name}...")
    model = AutoModel.from_pretrained(model_name)
    
    text = "আমি বাংলায় কথা বলতে পারি"
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    print(f"Output shape: {outputs.last_hidden_state.shape}")
    print("✅ SUCCESS: Model is publicly accessible and working.")
except Exception as e:
    print(f"❌ FAILED: {e}")
    raise e


In [ ]:
# ============================================================
# CELL 10 — Embedding Comparison Loop (Full Metrics)
# ============================================================
import os, time, json
import numpy as np
import pandas as pd
from experiments.runner import run_custom_vqc_experiment
from evaluation.metrics import compute_metrics, format_classification_report
from visualization.plots import (
    print_results_table, plot_model_comparison,
    plot_training_curves, plot_confusion_matrix,
)
from data.loader import load_dataset_for_language
from preprocessing.pipeline import preprocess_dataset
from embeddings.factory import get_embedder, embed_dataset
from models.custom_vqc.hybrid_model import CustomVQCClassifier

EMBEDDINGS_TO_TEST = ["tfidf", "mbert", "indicbert"]
VQC_VERSION = "v2"

# Output directories
for subdir in ["classification_reports", "confusion_matrices", "plots"]:
    os.makedirs(f"results/embedding_study/{subdir}", exist_ok=True)

embedding_results = {}      # {name: {test_metrics, history, params, ...}}
embedding_predictions = {}  # {name: {y_true, y_pred, texts}}
embedding_label_names = None

for emb_method in EMBEDDINGS_TO_TEST:
    print(f"\n{'='*65}")
    print(f"  EMBEDDING: {emb_method.upper()}")
    print(f"{'='*65}")

    try:
        # --- 1. Configure ---
        cfg.embeddings.method = emb_method
        cfg.custom_vqc.version = VQC_VERSION

        # --- 2. Load & embed data ---
        dataset = load_dataset_for_language(LANGUAGE, cfg, max_samples=MAX_SAMPLES)
        dataset = preprocess_dataset(dataset, cfg)
        embedder = get_embedder(cfg, LANGUAGE)
        embeddings = embed_dataset(embedder, dataset)

        X_train = embeddings["train"]
        y_train = np.array(dataset.train["labels"])
        X_val   = embeddings["val"]
        y_val   = np.array(dataset.val["labels"])
        X_test  = embeddings["test"]
        y_test  = np.array(dataset.test["labels"])
        input_dim  = X_train.shape[1]
        num_labels = dataset.num_labels
        label_names = dataset.label_names
        embedding_label_names = label_names

        print(f"  Input dim: {input_dim}, Classes: {num_labels}")
        print(f"  Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}")

        # --- 3. Train Custom VQC ---
        clf = CustomVQCClassifier(cfg, num_labels, input_dim, DEVICE)
        t0 = time.perf_counter()
        history = clf.train(X_train, y_train, X_val, y_val)
        training_time = time.perf_counter() - t0

        # --- 4. Evaluate ---
        test_metrics = clf.evaluate(X_test, y_test)
        y_pred = clf.predict(X_test)
        params = clf.get_params_count()

        # Store full results
        embedding_results[emb_method] = {
            "test_metrics": test_metrics,
            "history": history,
            "params": params,
            "training_time": training_time,
            "input_dim": input_dim,
        }

        # Store predictions for confusion matrix & sample analysis
        embedding_predictions[emb_method] = {
            "y_true": y_test,
            "y_pred": y_pred,
            "texts": dataset.test["texts"],
        }

        # --- 5. Print detailed metrics ---
        print(f"\n  Embedding: {emb_method.upper()}")
        print(f"  {'='*40}")
        print(f"  Accuracy:      {test_metrics['accuracy']:.4f}")
        print(f"  Precision:     {test_metrics['precision']:.4f}")
        print(f"  Recall:        {test_metrics['recall']:.4f}")
        print(f"  F1 Score:      {test_metrics['f1']:.4f}")
        print(f"  Training Time: {training_time:.1f}s")
        print(f"  Parameters:    {params:,}")

        # --- 6. Classification report ---
        report = format_classification_report(y_test, y_pred, label_names)
        print(f"\n  Classification Report:")
        print(report)

        report_path = f"results/embedding_study/classification_reports/{emb_method}_report.txt"
        with open(report_path, "w") as f:
            f.write(f"Embedding: {emb_method.upper()}\n")
            f.write(f"VQC: Custom VQC-{VQC_VERSION}\n")
            f.write(f"Language: {LANGUAGE}\n\n")
            f.write(report)
        print(f"  Saved: {report_path}")

    except Exception as e:
        print(f"  \u274c {emb_method} FAILED: {e}")
        import traceback; traceback.print_exc()
        embedding_results[emb_method] = {"error": str(e)}

print(f"\n{'='*65}")
print(f"  Embedding study loop complete!")
print(f"  Successful: {sum(1 for v in embedding_results.values() if 'error' not in v)}/{len(EMBEDDINGS_TO_TEST)}")
print(f"{'='*65}")


In [ ]:
# ============================================================
# CELL 11 — Results Table (Full Metrics)
# ============================================================
from visualization.plots import print_results_table

print_results_table(embedding_results, title="Embedding Comparison — Custom VQC-v2")

# Also print a compact summary with training time
print("\nDetailed Summary:")
print("-" * 90)
print(f"{'Embedding':<15} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Time (s)':>10} {'Params':>10}")
print("-" * 90)
for name, data in embedding_results.items():
    if isinstance(data, dict) and "test_metrics" in data:
        m = data["test_metrics"]
        print(f"{name:<15} {m['accuracy']:>10.4f} {m['precision']:>10.4f} {m['recall']:>10.4f} {m['f1']:>10.4f} {data.get('training_time',0):>10.1f} {data.get('params','?'):>10}")
    else:
        print(f"{name:<15} {'FAILED':>10}")
print("-" * 90)


In [ ]:
# ============================================================
# CELL 12 — Comparison Plots (All 5 Metrics)
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
from visualization.plots import plot_model_comparison

os.makedirs("results/embedding_study/plots", exist_ok=True)

# --- 1. F1 Plot ---
plot_model_comparison(
    embedding_results, metric="f1",
    title=f"Embedding Study — F1 Score ({LANGUAGE})",
    save_path="results/embedding_study/plots/embedding_f1.png"
)

# --- 2. Accuracy Plot ---
plot_model_comparison(
    embedding_results, metric="accuracy",
    title=f"Embedding Study — Accuracy ({LANGUAGE})",
    save_path="results/embedding_study/plots/embedding_accuracy.png"
)

# --- 3. Precision Plot ---
plot_model_comparison(
    embedding_results, metric="precision",
    title=f"Embedding Study — Precision ({LANGUAGE})",
    save_path="results/embedding_study/plots/embedding_precision.png"
)

# --- 4. Recall Plot ---
plot_model_comparison(
    embedding_results, metric="recall",
    title=f"Embedding Study — Recall ({LANGUAGE})",
    save_path="results/embedding_study/plots/embedding_recall.png"
)

# --- 5. Training Time Plot ---
fig, ax = plt.subplots(figsize=(10, 6))
names = []
times = []
for name, data in embedding_results.items():
    if isinstance(data, dict) and "training_time" in data:
        names.append(name.upper())
        times.append(data["training_time"])

if names:
    colors = sns.color_palette("Set2", len(names))
    bars = ax.bar(names, times, color=colors, edgecolor="black", linewidth=0.5)
    for bar, t in zip(bars, times):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{t:.1f}s", ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_ylabel("Training Time (seconds)")
    ax.set_title(f"Embedding Study — Training Time ({LANGUAGE})")
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig("results/embedding_study/plots/embedding_training_time.png", dpi=300, bbox_inches="tight")
    plt.show()

print("✅ All comparison plots saved to results/embedding_study/plots/")


In [ ]:
# ============================================================
# CELL 13 — Confusion Matrices
# ============================================================
from visualization.plots import plot_confusion_matrix

for emb_name, pred_data in embedding_predictions.items():
    print(f"\nConfusion Matrix — {emb_name.upper()}")
    plot_confusion_matrix(
        pred_data["y_true"],
        pred_data["y_pred"],
        label_names=embedding_label_names,
        title=f"Confusion Matrix — {emb_name.upper()}",
        save_path=f"results/embedding_study/confusion_matrices/{emb_name}_confusion_matrix.png",
    )

print("\n✅ Confusion matrices saved to results/embedding_study/confusion_matrices/")


In [ ]:
# ============================================================
# CELL 14 — Training Curves
# ============================================================
from visualization.plots import plot_training_curves

for emb_name, data in embedding_results.items():
    if isinstance(data, dict) and "history" in data:
        h = data["history"]
        if "train_loss" in h and len(h["train_loss"]) > 1:
            plot_training_curves(
                h,
                title=f"Training Curves — {emb_name.upper()}",
                save_path=f"results/embedding_study/plots/curves_{emb_name}.png"
            )

print("✅ Training curves saved.")


In [ ]:
# ============================================================
# CELL 15 — Sample Predictions (20 random examples per embedding)
# ============================================================
import pandas as pd
import random

all_sample_rows = []

for emb_name, pred_data in embedding_predictions.items():
    texts  = pred_data["texts"]
    y_true = pred_data["y_true"]
    y_pred = pred_data["y_pred"]

    n = min(20, len(texts))
    indices = random.sample(range(len(texts)), n)

    print(f"\n{'='*70}")
    print(f"  Sample Predictions — {emb_name.upper()} (20 random test examples)")
    print(f"{'='*70}")
    print(f"{'#':>3}  {'True Label':<15} {'Predicted':<15} {'Match':>5}  Text")
    print("-" * 70)

    for i, idx in enumerate(indices, 1):
        true_label = embedding_label_names[y_true[idx]] if embedding_label_names else str(y_true[idx])
        pred_label = embedding_label_names[y_pred[idx]] if embedding_label_names else str(y_pred[idx])
        match = "✅" if y_true[idx] == y_pred[idx] else "❌"
        text_snippet = texts[idx][:80].replace("\n", " ")
        print(f"{i:>3}  {true_label:<15} {pred_label:<15} {match:>5}  {text_snippet}")

        all_sample_rows.append({
            "embedding": emb_name,
            "text": texts[idx],
            "true_label": true_label,
            "predicted_label": pred_label,
            "correct": y_true[idx] == y_pred[idx],
        })

# Save all sample predictions to CSV
samples_df = pd.DataFrame(all_sample_rows)
samples_df.to_csv("results/embedding_study/sample_predictions.csv", index=False)
print(f"\n✅ Sample predictions saved to results/embedding_study/sample_predictions.csv")


In [ ]:
# ============================================================
# CELL 16 — Results CSV
# ============================================================
import pandas as pd

rows = []
for name, data in embedding_results.items():
    if isinstance(data, dict) and "test_metrics" in data:
        m = data["test_metrics"]
        rows.append({
            "Embedding": name,
            "Accuracy":  round(m["accuracy"], 4),
            "Precision": round(m["precision"], 4),
            "Recall":    round(m["recall"], 4),
            "F1":        round(m["f1"], 4),
            "Training_Time_s": round(data.get("training_time", 0), 1),
            "Parameters": data.get("params", "?"),
            "Input_Dim":  data.get("input_dim", "?"),
        })
    else:
        rows.append({
            "Embedding": name,
            "Accuracy": "FAILED",
            "Precision": "FAILED",
            "Recall": "FAILED",
            "F1": "FAILED",
            "Training_Time_s": "",
            "Parameters": "",
            "Input_Dim": "",
        })

results_df = pd.DataFrame(rows)
results_df.to_csv("results/embedding_study/embedding_results.csv", index=False)
print(results_df.to_string(index=False))
print(f"\n✅ Results CSV saved to results/embedding_study/embedding_results.csv")


In [ ]:
# ============================================================
# CELL 17 — Determine Best Embedding
# ============================================================
import json

# Primary criterion: F1. Secondary: Accuracy.
best_f1 = -1
best_acc = -1
BEST_EMBEDDING = "tfidf"  # fallback
best_reason = ""

for name, data in embedding_results.items():
    if isinstance(data, dict) and "test_metrics" in data:
        f1  = data["test_metrics"].get("f1", 0)
        acc = data["test_metrics"].get("accuracy", 0)
        if f1 > best_f1 or (f1 == best_f1 and acc > best_acc):
            best_f1  = f1
            best_acc = acc
            BEST_EMBEDDING = name
            if f1 > best_f1:
                best_reason = f"Highest F1 score ({f1:.4f})"
            else:
                best_reason = f"Tied F1 ({f1:.4f}), higher Accuracy ({acc:.4f})"

best_reason = f"Highest F1 score ({best_f1:.4f})" if best_reason == "" else best_reason

print(f"\n{'='*65}")
print(f"  BEST EMBEDDING SELECTION")
print(f"{'='*65}")
print(f"  Best Embedding: {BEST_EMBEDDING.upper()}")
print(f"  Reason:         {best_reason}")
print(f"  F1 Score:       {best_f1:.4f}")
print(f"  Accuracy:       {best_acc:.4f}")
print(f"{'='*65}")

# Comparison against all
print(f"\n  All Embeddings Ranked by F1:")
ranked = []
for name, data in embedding_results.items():
    if isinstance(data, dict) and "test_metrics" in data:
        ranked.append((name, data["test_metrics"]["f1"], data["test_metrics"]["accuracy"]))
    else:
        ranked.append((name, -1, -1))
ranked.sort(key=lambda x: (-x[1], -x[2]))
for rank, (name, f1, acc) in enumerate(ranked, 1):
    marker = " ← BEST" if name == BEST_EMBEDDING else ""
    if f1 >= 0:
        print(f"    {rank}. {name:<15} F1={f1:.4f}  Acc={acc:.4f}{marker}")
    else:
        print(f"    {rank}. {name:<15} FAILED")

# Save summary JSON
results_summary = {
    "best_embedding": BEST_EMBEDDING,
    "best_f1": best_f1,
    "best_accuracy": best_acc,
    "reason": best_reason,
    "language": LANGUAGE,
    "vqc_version": VQC_VERSION,
    "all_results": {
        k: v.get("test_metrics", {}) if isinstance(v, dict) else {"error": v.get("error", "unknown")}
        for k, v in embedding_results.items()
    }
}
with open("results/embedding_study/summary.json", "w") as f:
    json.dump(results_summary, f, indent=2, default=str)

print(f"\n✅ Summary saved to results/embedding_study/summary.json")
print(f"\n  Use BEST_EMBEDDING = \"{BEST_EMBEDDING}\" in notebook 05.")


---
## Interpretation

In [ ]:
# ============================================================
# CELL 18 — Auto-Generated Interpretation
# ============================================================
from IPython.display import display, Markdown

def generate_interpretation(results, best_name, label_names):
    """Generate a markdown interpretation of embedding study results."""
    lines = ["## Embedding Study — Interpretation\n"]

    # Collect successful results
    successful = {k: v for k, v in results.items() if isinstance(v, dict) and "test_metrics" in v}
    failed = [k for k, v in results.items() if isinstance(v, dict) and "error" in v]

    # 1. Best embedding
    best = successful.get(best_name, {})
    best_m = best.get("test_metrics", {})
    lines.append(f"### 1. Best Embedding: **{best_name.upper()}**\n")
    lines.append(f"- F1 Score: **{best_m.get('f1', 0):.4f}**")
    lines.append(f"- Accuracy: **{best_m.get('accuracy', 0):.4f}**")
    lines.append(f"- Training Time: {best.get('training_time', 0):.1f}s")
    lines.append(f"- Parameters: {best.get('params', '?'):,}\n")

    # 2. Transformer vs TF-IDF comparison
    tfidf_f1 = successful.get("tfidf", {}).get("test_metrics", {}).get("f1", 0)
    transformer_names = [k for k in successful if k != "tfidf"]
    lines.append("### 2. Transformer vs TF-IDF\n")
    if transformer_names:
        best_transformer = max(transformer_names, key=lambda k: successful[k]["test_metrics"]["f1"])
        bt_f1 = successful[best_transformer]["test_metrics"]["f1"]
        if bt_f1 > tfidf_f1:
            delta = bt_f1 - tfidf_f1
            lines.append(f"- Transformer embeddings ({best_transformer}) **outperform** TF-IDF by {delta:.4f} F1 points.")
            lines.append(f"- This suggests that contextual representations capture emotion-relevant features better than bag-of-words.\n")
        elif bt_f1 < tfidf_f1:
            delta = tfidf_f1 - bt_f1
            lines.append(f"- **TF-IDF outperforms** transformer embeddings by {delta:.4f} F1 points.")
            lines.append(f"- This may indicate that: (a) the extreme compression from 768-d to {N_QUBITS} qubits loses too much contextual information, ")
            lines.append(f"  or (b) TF-IDF's sparser, higher-dimensional features (5000-d) are easier for the linear compressor to project.\n")
        else:
            lines.append(f"- TF-IDF and transformer embeddings show **comparable** F1 scores.\n")
    else:
        lines.append("- No transformer embeddings completed successfully for comparison.\n")

    # 3. Accuracy vs F1 agreement
    lines.append("### 3. Accuracy vs F1 Agreement\n")
    for name, data in successful.items():
        m = data["test_metrics"]
        acc, f1 = m["accuracy"], m["f1"]
        if acc > f1 + 0.05:
            lines.append(f"- **{name}**: Accuracy ({acc:.4f}) > F1 ({f1:.4f}). "
                         f"This gap suggests the model performs well on majority classes but poorly on minority classes. "
                         f"Class imbalance is likely affecting performance.")
        elif f1 > acc + 0.05:
            lines.append(f"- **{name}**: F1 ({f1:.4f}) > Accuracy ({acc:.4f}). "
                         f"The weighted F1 compensates for class imbalance better than raw accuracy.")
        else:
            lines.append(f"- **{name}**: Accuracy ({acc:.4f}) and F1 ({f1:.4f}) are consistent.")
    lines.append("")

    # 4. Failed embeddings
    if failed:
        lines.append("### 4. Failed Embeddings\n")
        for name in failed:
            err = results[name].get("error", "Unknown error")
            lines.append(f"- **{name}**: Failed with `{err[:100]}`.")
        lines.append("")

    # 5. Recommendation
    lines.append("### 5. Recommendation for Phase B (VQC Ablation)\n")
    lines.append(f"- **Proceed with `{best_name}`** as the embedding for VQC architecture ablation.")
    lines.append(f"- Set `BEST_EMBEDDING = \"{best_name}\"` in notebook `05_vqc_ablation_study.ipynb`.")
    if best_name == "tfidf":
        lines.append("- Note: TF-IDF produces sparse 5000-d vectors. The linear compressor maps these to N qubits. "
                     "Consider also running Phase B with mBERT as a secondary comparison to validate the embedding choice.")

    return "\n".join(lines)

interpretation = generate_interpretation(embedding_results, BEST_EMBEDDING, embedding_label_names)
display(Markdown(interpretation))

# Save interpretation
with open("results/embedding_study/interpretation.md", "w") as f:
    f.write(interpretation)
print("\n✅ Interpretation saved to results/embedding_study/interpretation.md")


In [ ]:
# ============================================================
# CELL 19 — Save Results to Google Drive
# ============================================================
import os, shutil

results_src = os.path.join(PROJECT_DIR, "results", "embedding_study")
results_dst = os.path.join(DRIVE_PROJECT_DIR, "results", "embedding_study")

if os.path.exists(results_src):
    os.makedirs(results_dst, exist_ok=True)
    for root, dirs, files_list in os.walk(results_src):
        rel = os.path.relpath(root, results_src)
        dst = os.path.join(results_dst, rel)
        os.makedirs(dst, exist_ok=True)
        for f in files_list:
            shutil.copy2(os.path.join(root, f), os.path.join(dst, f))
    print(f"✅ Results saved to: {results_dst}")

    # List all saved files
    print("\nSaved files:")
    for root, dirs, flist in os.walk(results_dst):
        level = root.replace(results_dst, "").count(os.sep)
        indent = "  " * (level + 1)
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = "  " * (level + 2)
        for f in flist:
            print(f"{sub_indent}{f}")
else:
    print("⚠️  No results directory found — run experiments first.")
